In [1]:
# conda vis2
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent.parent.parent
exp_version = 'V08'
EXPERIMENT = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / exp_version / 'default'
ITERATIONS = EXPERIMENT / 'iterations'
INTERIM_DATA = PROJECT_ROOT / 'data' / 'interim'

import sys
sys.path.append(str(PROJECT_ROOT))

from src.utils.util import load_sensor_dict
# from src.main.run_at3d import load_config
from src.utils.util import load_config
import at3d
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
def obs_vec(sensor):
    vec = np.array([])
    for s in sensor['MISR']['sensor_list']:
        # condx = (s.ray_x.data >= 0.0) & (s.ray_x.data <= 17.6)
        # condy = (s.ray_y.data >= 0.0) & (s.ray_y.data <= 17.6)
        # cond = condx & condy
        # vec = np.append(vec, (np.pi * s.I.data[cond]) / s.mu0)
        vec = np.append(vec, (np.pi * s.I.data) / s.mu0)
    return vec

In [3]:
config = load_config(EXPERIMENT / 'config.yml')
iter_files = sorted(ITERATIONS.glob('iter*.nc'))
iterfile = iter_files[-2]
iter_index = int(iterfile.stem[4:])
obs_file = config['obs_file']
loss = pd.read_csv(EXPERIMENT / 'loss.csv')
target_chi2 = float(loss[' loss'][iter_index])

F_sensor = load_sensor_dict(iterfile)
y_sensor = load_sensor_dict(obs_file)

F = obs_vec(F_sensor)
y = obs_vec(y_sensor)

In [4]:
def chi2(y, F, S_1):
    m = len(y)
    return 0.5 * (1/m) * np.matmul(y-F, np.matmul(S_1, y-F))

In [5]:
## AT3D Error Cov
sf = config['error_cov_sf']
at3d_S_1 = np.diag(sf * np.ones(len(y)))
m = y_sensor.nmeasurements
at3d_chi2 = chi2(y, F, at3d_S_1)

In [6]:
print('Target chi^2  = {:.2f}'.format(target_chi2))
print('Calc chi^2    = {:.2f}'.format(at3d_chi2))
print('Calc / Target = {}'.format(at3d_chi2 / target_chi2))

Target chi^2  = 559.66
Calc chi^2    = 7171.12
Calc / Target = 12.81345761994287


In [7]:
# ## MISR Error Cov
misr_S_1 = np.diag(1 / (0.04 * y)**2)
misr_chi2 = chi2(y, F, misr_S_1)

print('chi^2 = {:.2f}'.format(misr_chi2))

chi^2 = 32.93
